In [3]:
!pip install -q git+https://github.com/zhouy185/rosa-starter.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [4]:
import numpy as np
from starter import ZONES, TIME_BLOCKS, COSTS, PROMISE, delivery_times

SEED = 1   # makes results the same every run

In [5]:
times = delivery_times("Far West", "Fri/Sat eve", 45)
print(len(times), "orders; average delivery time", round(times.mean(), 1),"minutes")

210 orders; average delivery time 43.2 minutes


## Part 1:

### Part 1(a): Percentage of late orders

#### A late order is one whose delivery time is bigger than the promise. The plan is to collect the delivery times for every zone and time block requested, then count how many are late.

In [6]:
def late_rate(zone, time_block, promise):
    # Step 1: decide which zones and time blocks to look at

    zones = ZONES if zone == "all" else [zone]
    time_blocks = TIME_BLOCKS if time_block == "all" else [time_block]

    # Step 2: count orders and late orders

    total_orders = 0
    late_orders = 0

    for z in zones:
        for t in time_blocks:
            times = delivery_times(z, t, promise, seed=SEED)

            total_orders = total_orders + len(times)
            late_orders = late_orders + (times > promise).sum()

    # Step 3: calculate percentage

    return late_orders / total_orders * 100


Testing the 4 required cases at 45 mintue promise

In [7]:
print("1. Specific zone + specific time block")
print(f"Far West + Fri/Sat eve: {late_rate('Far West', 'Fri/Sat eve', 45):.2f}%")

print("\n2. All zones + specific time block")
print(f"All zones + Lunch: {late_rate('all', 'Lunch', 45):.2f}%")

print("\n3. Specific zone + all time blocks")
print(f"Central + all time blocks: {late_rate('Central', 'all', 45):.2f}%")

print("\n4. All zones + all time blocks")
print(f"All zones + all time blocks: {late_rate('all', 'all', 45):.2f}%")

1. Specific zone + specific time block
Far West + Fri/Sat eve: 36.67%

2. All zones + specific time block
All zones + Lunch: 3.33%

3. Specific zone + all time blocks
Central + all time blocks: 4.23%

4. All zones + all time blocks
All zones + all time blocks: 8.29%


### Part 1 (b): Rank by late rate

In [8]:
late_rates = []

for zone in ZONES:
    for time_block in TIME_BLOCKS:
        rate = late_rate(zone, time_block, PROMISE)
        late_rates.append((zone, time_block, rate))

late_rates.sort(key=lambda x: x[2], reverse=True)

print("Rank by late rate:")
for zone, time_block, rate in late_rates:
    print(zone, "-", time_block, ":", round(rate, 2), "%")

Rank by late rate:
Far West - Fri/Sat eve : 36.67 %
North - Fri/Sat eve : 16.67 %
Far West - Weekday eve : 11.18 %
Central - Fri/Sat eve : 6.84 %
North - Weekday eve : 6.0 %
Far West - Other : 4.69 %
Central - Weekday eve : 4.24 %
North - Lunch : 3.68 %
Central - Lunch : 3.46 %
Central - Other : 2.63 %
North - Other : 2.5 %
Far West - Lunch : 2.5 %


### Part 1 (c): Average delivery time

In [9]:
def avg_time(zone, time_block, promise):

    # Step 1: decide which zones and time blocks to look at
    zones = ZONES if zone == "all" else [zone]
    time_blocks = TIME_BLOCKS if time_block == "all" else [time_block]

    # Step 2: count orders and minutes across all of them
    total_orders = 0
    total_minutes = 0

    for z in zones:
        for t in time_blocks:
            times = delivery_times(z, t, promise, seed=SEED)

            total_orders = total_orders + len(times)
            total_minutes = total_minutes + times.sum()

    # Step 3: Calculate average delivery time
    return total_minutes / total_orders


Testing the 4 required cases with 45 minute promise:

In [10]:
print("1. Specific zone + specific time block")
print(f"Far West + Fri/Sat eve: {avg_time('Far West', 'Fri/Sat eve', 45):.2f} minutes")

print("\n2. All zones + specific time block")
print(f"All zones + Lunch: {avg_time('all', 'Lunch', 45):.2f} minutes")

print("\n3. Specific zone + all time blocks")
print(f"Central + all time blocks: {avg_time('Central', 'all', 45):.2f} minutes")

print("\n4. All zones + all time blocks")
print(f"All zones + all time blocks: {avg_time('all', 'all', 45):.2f} minutes")

1. Specific zone + specific time block
Far West + Fri/Sat eve: 42.92 minutes

2. All zones + specific time block
All zones + Lunch: 28.22 minutes

3. Specific zone + all time blocks
Central + all time blocks: 28.45 minutes

4. All zones + all time blocks
All zones + all time blocks: 31.19 minutes


### Part 1 (d): Rank by Average time

In [11]:
average_times = []

for zone in ZONES:
    for time_block in TIME_BLOCKS:
        average = avg_time(zone, time_block, PROMISE)
        average_times.append((zone, time_block, average))

average_times.sort(key=lambda x: x[2], reverse=True)

print("Rank by average delivery time:")
for zone, time_block, average in average_times:
    print(zone, "-", time_block, ":", round(average, 2), "minutes")

Rank by average delivery time:
Far West - Fri/Sat eve : 42.92 minutes
North - Fri/Sat eve : 36.77 minutes
Far West - Weekday eve : 35.04 minutes
Central - Fri/Sat eve : 32.13 minutes
North - Weekday eve : 31.38 minutes
Far West - Lunch : 30.58 minutes
Far West - Other : 30.04 minutes
Central - Weekday eve : 29.03 minutes
North - Lunch : 28.67 minutes
North - Other : 27.66 minutes
Central - Lunch : 26.8 minutes
Central - Other : 26.04 minutes


### Part 1(e): Which measure matters more?

- The late rate is more relevant to Rosa. Rosa's costs come from late orders: each one triggers a refund and makes the customer order less in the future. The promise is 45 minutes, so a customer only cares whether the pizza arrived within 45 minutes, not whether the average is 30.

- The average hides the problem. Far West Fri/Sat eve has an average of 42.9 minutes, which is under 45 and looks fine, but 36.7% of its orders are late. A few very slow deliveries push many orders past the promise while the average still looks okay.

- The two rankings also disagree in the middle. Far West Lunch is 6th by average time (30.6 min) but has only a 2.5% late rate, tied for the lowest. Far West "Other" has a higher late rate (4.7%) but a lower average. So the average would send Rosa's attention to the wrong places. The average is still useful as a secondary signal, because slower averages mean orders are closer to being late.

## Part 2: Best Promise

### Part 2(a): Cost per late order

In [12]:
print(COSTS)

{'refund': 10.0, 'churn_orders': 1.8, 'margin': 9.0}


In [13]:
def cost_per_late_order(costs):
    return costs["refund"] + costs["churn_orders"] * costs["margin"]

print("Total cost per late order: $", cost_per_late_order(COSTS))

Total cost per late order: $ 26.2


### Part 2(b): Choosing the range

In [14]:
# From part a
def cost_per_late_order(costs):
    return costs["refund"] + costs["churn_orders"] * costs["margin"]

# Part II b
def best_promise(zone, time_block, promises, costs):
    late_cost = cost_per_late_order(costs)

    best_promise_so_far = None
    best_profit_so_far = None

    for p in promises:
        # Get every delivery time for this promise
        times = delivery_times(zone, time_block, p, seed=SEED)

        number_of_orders = len(times)
        number_of_late = (times > p).sum()

        # Net profit = money from all orders - cost of late orders
        profit = number_of_orders * costs["margin"] - number_of_late * late_cost

        # Keep this promise if it beats the best one so far
        if best_profit_so_far is None or profit > best_profit_so_far:
            best_promise_so_far = p
            best_profit_so_far = profit

    return best_promise_so_far, best_profit_so_far

Test it:

In [15]:
promises_to_try = list(range(20, 85, 5))   # 20, 25, 30, ..., 80

p, profit = best_promise("Far West", "Fri/Sat eve", promises_to_try, COSTS)
print("Recommended promise:", p, "minutes")
print("Net profit: $", round(profit, 2))

Recommended promise: 55 minutes
Net profit: $ 1212.4


Why range 20 to 80:

- A short promise gets more orders, but almost all of them are late, so profit is negative. Around 20 minutes is already clearly a loss, so there's no need to go lower.

- A long promise has almost no late orders, but customers order less, so profit falls again. At 65+ minutes for Far West Fri/Sat eve, there are almost no late orders, and profit only shrinks from there.

- The range must go past 50. If I had stopped at 50, I would have picked 50 minutes ($827). The true best is 55 minutes ($1,212), which would have been missed.

- General rule: choose a range wide enough that the best answer lands in the middle, not at the edge. If the winner is the first or last value tried, widen the range and try again.

## Overall Recommendation

- Based on the analysis, the delivery promise should not be chosen using only the average delivery time. The late-order rate is more directly connected to Rosa's costs because a late order results in a refund and expected future lost profit from customer churn.

- For the best-promise analysis, I tested promised delivery times from 20 to 80 minutes in 5-minute increments. This range is wide enough to include both relatively short promises, which generate more late orders, and longer promises, which reduce the number of customers ordering.

- For the example of **Far West during Fri/Sat evening**, the analysis recommends a **55-minute delivery promise**, with a calculated net profit of **$1,212** using the assumptions and simulation seed used in this notebook.

- The result also shows why the promise should be evaluated using net profit rather than simply minimizing late orders. A longer promise can reduce late orders, but it can also reduce demand. The recommended promise balances these two effects by maximizing net profit.

- The same `best_promise()` function can be reused in the Streamlit app so that Rosa can select a different zone, time block, promise range, or cost assumption and calculate the recommendation dynamically.


## Streamlit App and GitHub Repository Links

**GitHub repository:**
https://github.com/Shriram-2001/Rosa-Pizza-Assignment1

**Streamlit Community Cloud app:**
https://rosa-pizza-assignment1-isznqghsmcnkmmgh9gzgsj.streamlit.app


# Appendix: Use of AI

## How I Used AI

I used AI as a learning and development assistant during this assignment. I used it to understand Python concepts, debug code, explain errors, review my approach, and help convert the notebook analysis into a Streamlit application.

I did not use AI to replace the analysis. I provided my assignment requirements and my own code and used AI to help me understand and improve the implementation. I also checked that the final calculations and logic matched the requirements of the assignment.

For the Streamlit portion, I used GitHub Copilot to assist with creating the application. I created a project-specific skill file at `.github/skills/notebook-to-streamlit/SKILL.md` and explicitly instructed Copilot to follow this skill and reuse the analytical logic developed in the notebook.

## Key Prompts Used

### 1. Understanding and debugging the Python analysis

I asked AI to explain my Python code in simple terms and help me identify and fix errors when they occurred.

**Key prompt:**

> Explain this Python code line by line in simple terms and help me debug it if there is an error.

### 2. Developing the best-promise calculation

I provided my Part II requirements and existing functions and asked AI to help implement the best-promise calculation without changing the underlying assignment logic.

**Key prompt:**

> Help me write a reusable function that tests promised delivery times in 5-minute increments and calculates net profit using the number of orders, late orders, profit margin, refund cost, and churn cost. Use my existing `cost_per_late_order()` function and `delivery_times()` function.

### 3. Creating the Streamlit application

For Part III, I asked GitHub Copilot to use the project-specific skill and the notebook as the source of the analytical logic.

**Key prompt:**

> Before writing any code, read and follow `.github/skills/notebook-to-streamlit/SKILL.md`. Also inspect my existing notebook, `Assignment_1_Rosa_Pizza.ipynb`. The notebook is the source of truth for the analytical logic. Use the logic I already developed in Part II(b) rather than creating a different calculation. Create an `app.py` file for the Streamlit application.

### 4. Reviewing the Streamlit application

After creating the app, I asked Copilot to check that the application met the assignment requirements.

**Key prompt:**

> Review the generated `app.py` against the requirements in `.github/skills/notebook-to-streamlit/SKILL.md` and Part III of my assignment. Check that the zone and time block use dropdowns, the promise range is adjustable, profit margin, churn, and refund cost are adjustable, the Part II(b) logic is reused, promises are tested in 5-minute increments, and the recommendation is calculated dynamically rather than hard-coded.

## Role of AI in the Assignment

AI was mainly used for explanation, debugging, code development, and reviewing the Streamlit implementation. The final notebook contains the analysis, calculations, reasoning, and recommendations used for the assignment.
